# 3 · The agent: how one user message flows

The agent (Google ADK) is the **orchestrator and router**: it understands the message and picks a tool. Everything
that must hold every time is code around it:

```
user message
  └─► start_turn (code)          patient check: a classifier run by code (doubt → a person); resets the turn
       └─► router (Gemini)        which kind of question? which tool, with which arguments?
            └─► limit_tool_calls (code)   ≤ 3 calls, no repeats; a patient case may only escalate
                 └─► tool (code)   search_current · search_historical · lookup_document · compare_studies
                                   · escalate (functions) · sql_analyst (a specialist agent)
       └─► relay_tool_answer (code)  the user sees the tool's checked answer WORD FOR WORD, never the model's words
```

`LIVE = False` runs every callback and tool by hand on recorded data (no cost). The last section runs a real
conversation through the ADK runner and needs `LIVE = True` (the router itself is a Gemini call).

In [1]:
LIVE = False   # False: recorded responses and local files, no cost. True: real Google Cloud calls (small cost).

from nb_setup import setup, short, table, module, functions
SUBMISSION = setup(live=LIVE)

submission : /Users/vikassingh/Downloads/GIT/GCP_FDE/Session-08-UseCases-Assignments/submission
mode       : OFFLINE: recorded responses, no cost
project    : tredence-fde-1 · answers: gemini-3.5-flash-lite · agent: gemini-3.1-flash-lite · embeddings: gemini-embedding-001


## The agent and its parts (`harness/medevidence_agent_vks/agent.py`)

In [2]:
from medevidence_agent_vks import agent
import tools, sql_agent, patient_check

ra = agent.root_agent
print("name     :", ra.name)
print("model    :", ra.model.model if hasattr(ra.model, "model") else ra.model)
print("tools    :", [getattr(t, "name", getattr(t, "__name__", "")) for t in ra.tools])
print("callbacks: before_agent =", ra.before_agent_callback.__name__, "· before_tool =", ra.before_tool_callback.__name__,
      "· after_model =", ra.after_model_callback.__name__, "· on_model_error =", ra.on_model_error_callback.__name__)
print("\nthe instruction's rules (it is built from the registry at start-up, not written by hand):")
for line in agent.INSTRUCTION.splitlines():
    if line[:3].strip().rstrip(".").isdigit():
        print("  ", line.strip()[:110])

name     : medevidence_agent_vks
model    : gemini-3.1-flash-lite
tools    : ['search_current', 'search_historical', 'lookup_document', 'compare_studies', 'sql_analyst', 'escalate']
callbacks: before_agent = start_turn · before_tool = limit_tool_calls · after_model = relay_tool_answer · on_model_error = busy_fallback

the instruction's rules (it is built from the registry at start-up, not written by hand):
   1. CONVERSATION, NO TOOL: reply naturally and briefly yourself to
   2. THE COLLECTION ITSELF, NO TOOL: "what is <product>?", "what documents do you have?", "which
   3. A DOCUMENT ID IS NAMED (an ID from the CATALOGUE, or one shaped like them): call lookup_document.
   4. CURRENT GUIDANCE WITH A NAMED COUNTRY (what applies now): call search_current with the user's
   5. EARLIER GUIDANCE WITH A NAMED COUNTRY: call search_historical (product as in 4).
   6. NO COUNTRY NAMED for 4 or 5: ask "For which country?" and list the covered ones: Germany, Singapore, UAE, U
   7. COMPARING ST

In [3]:
functions(agent)

,line,function,what it does
0,174,unsupported_clinical_terms,D56: clinical words in a no-tool reply that ne...
1,186,relay_tool_answer,"What the user sees, decided in code."
2,242,start_turn,Each user message starts a new turn: no relaye...
3,260,limit_tool_calls,D51: at most MAX_TOOL_CALLS_PER_TURN tool call...
4,298,busy_fallback,Still busy after the retries: a polite message...


## 1 · Patient check (D70, D73): a classifier run by code, any doubt goes to a person
A small Gemini classifier runs **by code** before the router sees the message ("is this a treatment decision for one
person: yes / no / unsure?"). Yes, "unsure", or a failed call makes it a patient case; nothing can undo that. (The
word patterns that used to run first were dropped in D73: the classifier catches everything they caught.)

In [4]:
for message in ["My 70-year-old patient has eGFR 35. What dose should I give him?",
                "Mein Patient hat eine eGFR von 35. Welche Dosis soll ich geben?",
                "What does the label say for eGFR 30-59?"]:
    v = patient_check.check(message)
    print(f"patient case: {str(v['patient_case']):5} · classifier: {v['classifier']:7} · {short(message, 60)} · {short(v['reason'], 70)}")

patient case: True  · classifier: yes     · My 70-year-old patient has eGFR 35. What dose should I give  … · The user is asking for a specific dosage recommendation for a 70-year- …
patient case: True  · classifier: yes     · Mein Patient hat eine eGFR von 35. Welche Dosis soll ich geb … · The user is asking for a specific dosage recommendation for a patient  …
patient case: False · classifier: no      · What does the label say for eGFR 30-59? · The query asks for general information regarding a specific patient ca …


## 2 · `start_turn` and `limit_tool_calls`: rules the model cannot skip

In [5]:
from types import SimpleNamespace
from google.genai import types

def turn(text):   # what ADK hands the callback: the session state and the user's message
    ctx = SimpleNamespace(state={}, user_content=types.Content(role="user", parts=[types.Part(text=text)]))
    agent.start_turn(ctx)
    return ctx

ctx = turn("Current AX-17 renal guidance in Germany?")
print("state after start_turn:", {k: v for k, v in ctx.state.items() if not k.startswith("_")})
search = SimpleNamespace(name="search_current")
print("\n1st call            →", agent.limit_tool_calls(search, {"question": "renal", "country": "Germany"}, ctx))
print("same call again     →", agent.limit_tool_calls(search, {"question": "renal", "country": "Germany"}, ctx))
p = turn("Mein Patient hat eine eGFR von 35. Welche Dosis soll ich geben?")
print("patient case, search →", agent.limit_tool_calls(search, {"question": "dose"}, p))
print("patient case, escalate →", agent.limit_tool_calls(SimpleNamespace(name="escalate"), {"reason": "patient_specific"}, p))

state after start_turn: {'relay_answer': [], 'relay_sql': [], 'tool_calls_this_turn': [], 'user_text_this_turn': 'Current AX-17 renal guidance in Germany?', 'patient_case_this_turn': False, 'patient_check_this_turn': {'patient_case': False, 'classifier': 'no', 'reason': 'The query asks for general guidance for a category of patients rather than advice for a specific individual.'}, 'escalated_this_turn': False, 'country_previous_turn': None, 'country_this_turn': None}

1st call            → None
same call again     → {'outcome': 'skipped', 'answer': '', 'reason': 'the same call was already made this turn; its answer will be shown'}
patient case, search → {'outcome': 'skipped', 'answer': '', 'reason': 'patient-specific question: call escalate with reason patient_specific instead'}
patient case, escalate → None


## 3 · A tool, called by hand: what the agent gets back vs what the user sees
`search_current` runs `rt.answer()` (notebook 2), stores the exact reply for the user in the session state, and gives
the router only a compact summary.

In [6]:
ctx = turn("Current AX-17 renal guidance in Germany?")
back = tools.search_current("Current AX-17 renal guidance in Germany?", "Germany", "", ctx)
print("returned to the router:", {k: (short(v, 80) if isinstance(v, str) else v) for k, v in back.items() if k != "evidence"})
print("\nstored for the user (relay):\n")
print(ctx.state[tools.RELAY_KEY][0][:1200])

returned to the router: {'outcome': 'answered', 'answer': 'For normal renal function (eGFR >=90), the starting dose is 50 mg OD with a maxi …', 'sources': 'Sources ⏎ [E1] SB-AX17-2026-09 · SB AX17 2026 09 Safety Bulletin v1.0 · APPROVED …', 'filters_applied': {'product': 'AX-17', 'status': ['APPROVED'], 'boilerplate': False, 'embedding_model': 'gemini-embedding-001@768', 'in_effect_on': '2026-10-20T14:25+00:00', 'country': 'Germany (+ GLOBAL clinical studies, labelled)'}, 'reasons': []}

stored for the user (relay):

For normal renal function (eGFR >=90), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine monitoring [E2]. For mild renal impairment (eGFR 60-89), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine monitoring [E2]. For moderate renal impairment (eGFR 30-59), the starting dose is 25 mg OD with a maximum of 50 mg/day, and monitoring requires Recheck 2-4 weeks [E1, E2, E3]. For severe renal impairment (eGFR 15-29), the starting dose is 

In [7]:
from google.adk.models import LlmResponse

# After the tool ran, the model writes its own text; the after_model callback replaces it with the relay, word for word.
model_text = LlmResponse(content=types.Content(role="model", parts=[types.Part(text="In short: 25 mg, I think.")]))
shown = agent.relay_tool_answer(ctx, model_text)
print("the model wrote: 'In short: 25 mg, I think.'")
print("the user sees  :", short(shown.content.parts[0].text, 200))

the model wrote: 'In short: 25 mg, I think.'
the user sees  : For normal renal function (eGFR >=90), the starting dose is 50 mg OD with a maximum of 100 mg/day and routine monitoring [E2]. For mild renal impairment (eGFR 60-89), the starting dose is 50 mg OD wit …


## 4 · `escalate`: the hand-off to a person
A record goes to the `escalations` queue (offline: `output/escalations_offline.jsonl`) and the user gets a fixed
hand-off text with the escalation ID. No dose for an individual is ever produced.

In [8]:
ctx = turn("My 70-year-old patient has eGFR 35. What dose should I give him?")
back = tools.escalate("patient_specific", "My 70-year-old patient has eGFR 35. What dose should I give him?", "", "", ctx)
print(back["outcome"], "\n")
print(ctx.state[tools.RELAY_KEY][0])

escalated 

Individual treatment decisions are for the treating physician. I've routed your question to Medical Information (escalation ESC-E2C0A9B8); they will follow up.


## 5 · The SQL specialist's tools (`harness/sql_agent.py`)
`sql_analyst` is an agent because writing SQL needs a loop (write → try → read the error → fix). Its **tools are code**:
`try_sql` checks the query (SELECT only, the two curated views only, a BigQuery dry run of the real tables), runs it and
previews; `show_sql_result` shows the last successful try, checked again, word for word.

In [9]:
import config, text2sql

sctx = turn("Which study had the highest rash rate?")
sql_agent.start_sql(sctx)
view = text2sql.view_ids()["v_adverse_events"]
sql = f"SELECT study_id, treatment_rate_pct FROM `{view}` WHERE event = 'rash' ORDER BY treatment_rate_pct DESC LIMIT 1"
print("try_sql        →", sql_agent.try_sql(sql, sctx))
print("show_sql_result →", sql_agent.show_sql_result(sctx))
print("\nshown to the user:\n" + sctx.state[sql_agent.SQL_RELAY_KEY][0])
print("\na forbidden query →", sql_agent.try_sql(f"SELECT chunk_id FROM `{config.table_id('chunks')}` LIMIT 1", sctx))

try_sql        → {'ok': True, 'rows_total': 1, 'preview': [{'study_id': 'HN-402', 'treatment_rate_pct': '19.7'}], 'hint': ''}
show_sql_result → {'ok': True, 'rows_total': 1}

shown to the user:
Result of a read-only query on v_adverse_events (1 row):

| study_id | treatment_rate_pct |
|-|-|
| HN-402 | 19.7 |

SQL:
SELECT study_id, treatment_rate_pct FROM `tredence-fde-1.medevidence_v1_vks.v_adverse_events` WHERE event = 'rash' ORDER BY treatment_rate_pct DESC LIMIT 1

a forbidden query → {'ok': False, 'error': 'only the curated views may be read, not tredence-fde-1.medevidence_v1_vks.chunks'}


## 6 · A real conversation through the ADK runner (LIVE)

`InMemoryRunner` is what `adk web` uses underneath: a session holds the state, and every step comes back as an
**event** (a function call, a function response, the final text). Below, each event is printed as it happens.

In [10]:
from google.adk.runners import InMemoryRunner

async def converse(messages):
    runner = InMemoryRunner(agent=agent.root_agent, app_name="notebook")
    session = await runner.session_service.create_session(app_name="notebook", user_id="notebook")
    for text in messages:
        print("=" * 100, "\nUSER:", text)
        message = types.Content(role="user", parts=[types.Part(text=text)])
        async for event in runner.run_async(user_id="notebook", session_id=session.id, new_message=message):
            for call in event.get_function_calls():
                print(f"  → tool call   {call.name}({', '.join(f'{k}={v!r}' for k, v in (call.args or {}).items())})")
            for resp in event.get_function_responses():
                body = resp.response or {}   # function tools return a dict with 'outcome'; an agent tool returns its text as 'result'
                status = body.get("outcome") or ("ok" if body.get("ok") else None) or short(body.get("result", ""), 60)
                print(f"  ← tool result {resp.name}: {status}")
            if event.is_final_response() and event.content and event.content.parts:
                text_out = "".join(p.text or "" for p in event.content.parts if not getattr(p, "thought", False))
                print("\nSHOWN TO THE USER:\n" + text_out)

MESSAGES = ["Current AX-17 renal guidance in Germany?",                       # current guidance → search_current
            "German AX-17 guidance before latest revision?",                  # then vs now → both tools (D71)
            "Mein Patient hat eine eGFR von 35. Welche Dosis soll ich geben?", # patient case (D70) → escalate
            "Which study had the highest rash rate?"]                         # numbers → sql_analyst
if LIVE:
    await converse(MESSAGES)
else:
    print("Set LIVE = True in the first cell to run this conversation (the router is a Gemini call; ~1 cent per message).")

Set LIVE = True in the first cell to run this conversation (the router is a Gemini call; ~1 cent per message).


## The agent layer on Databricks

| Here (Google ADK) | On Databricks |
|---|---|
| `Agent` (root, orchestrator + router) with function tools | Mosaic AI Agent Framework (an MLflow `ResponsesAgent` / `ChatAgent`, often written with LangGraph) |
| tools = Python functions | Unity Catalog functions, or Python tools in the agent code |
| `AgentTool(sql_analyst)` | a sub-agent / Genie space for SQL questions |
| callbacks (`start_turn`, `limit_tool_calls`, `relay_tool_answer`) | the same logic as nodes / guard steps in the graph (it is plain Python) |
| `adk web` / `InMemoryRunner` | the AI Playground / Review App; Model Serving to deploy |
| `agent_log`, `escalations` (BigQuery) | MLflow tracing + inference tables (Delta) |

The rule carries over unchanged: **the model may choose the path, never the facts.** Routing is the model's job;
filters, checks and what the user sees are code.